In [15]:
# !pip install transformers kobert-transformers

In [1]:
import re
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:

# ============================================================
# 1. 텍스트 정규화 함수
# ============================================================
def normalize_korean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ .,!?\"'’‘…~\-]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ============================================================
# 2. 데이터 로드 및 전처리
# ============================================================
data_path = "data/ratings_train.txt"  # 경로 확인 필요

df = pd.read_csv(data_path, sep="\t").head(1000)
df = df.dropna(subset=["document"])
df["document"] = df["document"].apply(normalize_korean_text)
df = df[df["document"].str.len() > 1]  # 1자 이하 제거
df.drop_duplicates('document', inplace=True)
df["label"] = df["label"].astype("int64")

print("샘플 확인:")
print(df.sample(5))


샘플 확인:
          id                                           document  label
576  2956430                                   너무나 따뜻하고 감동적인 영화      1
457  3398398                                 ㅋㅋㅋㅋㅋㅋㅋㅋ 반도 안되는 영화      0
915  8386647  유쾌함과 스릴러적인 요소가 잘 어울려진 명작. 후속작 보다 훨씬 매력있음. 두고 두...      1
208  1025241                     화려한여정이인상깊어요ㅋㅋ재밋어요ㅋㅋ배두나연기정말잘해요ㅋ      1
928  7457162                                나홀로집에와더불어 나의최고의영화..      1


In [ ]:

# train/test 분리
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Hugging Face의 Dataset 객체로 변환 
# BERT 모델에서 바로 사용이 가능한 객체의 형태
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds

Dataset({
    features: ['id', 'document', 'label'],
    num_rows: 796
})

In [ ]:

# ============================================================
# 3. KoBERT 모델 + Tokenizer
# ============================================================
MODEL_NAME = "skt/kobert-base-v1"
# AutoTokenizer를 이용해 KoBERT 전용 토크나이저 불러오기
# use_fast=False : KoBERT는 sentencepiece 기반이라 fast 버전(WordPiece용)이 아닌 slow 버전을 사용해야 함
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tok_fn(batch):
    """
    각 데이터(batch)의 'document' 컬럼(텍스트)을 KoBERT 토크나이저로 변환하는 함수.

    truncation=True  → 문장이 모델 최대 입력 길이를 초과하면 자동으로 자름
    max_length=128   → 최대 토큰 길이를 128로 고정 (BERT 입력 시 일반적인 설정)
    return 값에는 input_ids, attention_mask, token_type_ids 등이 포함됨
    """
    return tokenizer(batch["document"], truncation=True, max_length=128)
# ----------------------------------------------------------------------
# (2) 학습 데이터셋(train_ds) 토크나이징
# ----------------------------------------------------------------------
# Dataset.map() :
#   - 데이터셋의 각 샘플에 tok_fn 함수를 적용해 토큰화된 결과를 추가
#   - batched=True → 여러 샘플을 한 번에 처리 (속도 향상)
#   - remove_columns=["id", "document"] → 원본 텍스트 컬럼 제거 (모델에는 필요 없음)
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=["id", "document"])
test_tok  = test_ds.map(tok_fn,  batched=True, remove_columns=["id", "document"])


Map: 100%|██████████| 200/200 [00:00<00:00, 18228.97 examples/s]


In [24]:
train_ds['document']

Column(['너무나 따뜻하고 감동적인 영화', '제대하고 보니까 더 재밌네요 ㅋㅋㅋㅋ', '오오 난 여태까지 맷데이먼의 최고의영화는 본 얼티메이텀,라이언일병구하기 뿐인줄알았음ㅠ 굿윌헌팅도쩐당', '서스펜스도 없고 스릴도 없다.', '재밋네요 달팽이가 빨라서 더 재밌었어요'])

In [26]:
train_tok['input_ids']
# {
#   'input_ids': [2, 2743, 5755, 7098, 6924, 54, 3, 0, 0, 0, ...],  # 문장 토큰의 숫자 인덱스
#   'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, ...],             # 문장 구분용 (단일문장은 0, 두번째 문장이 존재하면 1, 2는 버그(무시 가능))
#   'attention_mask': [1, 1, 1, 1, 1, 1, 1, 0, 0, ...],             # 실제 토큰=1, 패딩=0
#   'label': 1                                                      # 감정 레이블
# }

Column([[1458, 5655, 1833, 5997, 7788, 791, 7206, 3394, 3, 2], [4128, 5808, 7788, 2355, 5771, 1698, 3969, 6269, 5703, 517, 492, 492, 492, 492, 3, 2], [3417, 6964, 1406, 3298, 7598, 5592, 517, 6176, 5850, 7096, 6184, 7095, 4524, 6954, 5760, 2408, 517, 6870, 7673, 6191, 7614, 46, 6011, 6865, 7126, 6361, 5495, 7789, 2571, 7119, 7292, 6821, 6827, 7089, 0, 517, 5520, 7059, 7890, 7681, 5859, 0, 5804, 3, 2], [2718, 6664, 7716, 6664, 5859, 3271, 2929, 6135, 5859, 3273, 54, 3, 2], [3969, 6268, 5703, 1597, 7705, 7096, 5330, 2562, 6003, 6553, 1698, 3969, 6269, 6885, 6857, 3, 2]])

In [ ]:

# ============================================================
# 🧠 BERT 분류 모델 (BERT + Linear Head)
# ============================================================
class BertClsHead(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        # nn.Module을 상속받은 사용자 정의 모델 초기화
        super().__init__()

        # ------------------------------------------------------------
        # (1) 사전학습된 BERT 모델 로드 (백본)
        # ------------------------------------------------------------
        # BertModel: 사전학습된 BERT Encoder (Transformer 층)
        # from_pretrained(model_name): 지정한 이름의 모델 가중치와 설정을 로드
        self.backbone = BertModel.from_pretrained(model_name)

        # BERT의 히든 벡터 차원 수 (예: 768)
        hidden = self.backbone.config.hidden_size

        # ------------------------------------------------------------
        # (2) Dropout과 Linear Classifier Head 정의
        # ------------------------------------------------------------
        # Dropout: 과적합 방지를 위해 일부 뉴런을 랜덤으로 0으로 만드는 정규화 기법
        self.dropout = nn.Dropout(dropout)

        # Linear Layer: BERT의 출력(768차원)을 num_labels(분류 클래스 수)로 변환
        self.classifier = nn.Linear(hidden, num_labels)

        # BERT의 PAD 토큰 ID를 명시적으로 설정 (마스크 연산 시 안정성 확보)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id


    # ------------------------------------------------------------
    # (3) 순전파(forward) 정의
    # ------------------------------------------------------------
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        """
        input_ids: 토큰화된 입력 문장 (Tensor)
        attention_mask: 실제 단어=1 / 패딩=0 으로 구분하는 마스크
        labels: 학습 시 정답 라벨 (없으면 추론 모드)
        """

        # ① BERT 백본에 입력 전달
        # out.last_hidden_state : 각 토큰의 벡터 (batch_size, seq_len, hidden_size)
        # out.pooler_output : [CLS] 토큰을 별도로 가공한 벡터 (여기서는 직접 CLS 사용)
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        # ② [CLS] 토큰 벡터 추출
        # BERT 입력의 첫 번째 토큰([CLS])은 문장 전체를 대표하는 의미로 학습됨
        # 문장 벡터 데이터를 의미
        pooled = out.last_hidden_state[:, 0]  # 첫 번째 토큰([CLS]) 위치의 벡터 선택

        # ③ Dropout + Linear Layer를 통과시켜 분류 로짓(logits) 계산
        logits = self.classifier(self.dropout(pooled))

        # 결과 딕셔너리 초기화
        result = {"logits": logits}

        # ④ 학습 단계: 정답 레이블이 있으면 손실(loss) 계산
        if labels is not None:
            # CrossEntropyLoss: 다중분류용 손실함수 (logits vs 정답 비교)
            loss = nn.CrossEntropyLoss()(logits, labels)
            result["loss"] = loss

        # ⑤ 추론 단계에서는 logits만 반환
        return result


# ------------------------------------------------------------
# (4) 모델 인스턴스 생성
# ------------------------------------------------------------
# MODEL_NAME: 사전학습된 KoBERT 이름 (예: "skt/kobert-base-v1")
# num_labels=2 : 긍정/부정 2클래스 분류
model = BertClsHead(MODEL_NAME, num_labels=2)


In [28]:

# ============================================================
# 5. 평가 함수
# ============================================================
def metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pred), "f1": f1_score(y, pred)}


In [29]:
# ============================================================
# 6. 학습 설정
# ============================================================

# TrainingArguments : Trainer가 학습할 때 사용할 각종 설정값을 정의하는 객체
args = TrainingArguments(
    # --------------------------------------------------------
    # (1) 출력 디렉토리 설정
    # --------------------------------------------------------
    output_dir="./kobert_from_bertmodel",   # 학습 결과(모델, 로그 등)를 저장할 경로

    # --------------------------------------------------------
    # (2) 배치 크기 설정
    # --------------------------------------------------------
    per_device_train_batch_size=16,         # GPU/CPU 하나당 학습 시 배치 크기
    per_device_eval_batch_size=16,          # 평가 시 배치 크기

    # --------------------------------------------------------
    # (3) 평가 및 저장 주기 설정
    # --------------------------------------------------------
    eval_strategy="epoch",                  # 한 epoch마다 평가 수행
    save_strategy="epoch",                  # 한 epoch마다 모델 저장

    # --------------------------------------------------------
    # (4) 학습 관련 하이퍼파라미터
    # --------------------------------------------------------
    num_train_epochs=2,                     # 학습 epoch 수 (전체 데이터 반복 횟수)
    learning_rate=5e-5,                     # AdamW 옵티마이저의 학습률
    weight_decay=0.01,                      # L2 정규화(가중치 감쇠) 계수
    warmup_ratio=0.1,                       # 학습 초기에 LR을 천천히 올리는 비율 (10%)
    logging_steps=50,                       # 로그를 출력할 step 간격

    # --------------------------------------------------------
    # (5) 모델 선택 및 저장 기준
    # --------------------------------------------------------
    load_best_model_at_end=True,            # 학습이 끝나면 가장 성능 좋은 모델 자동 로드
    metric_for_best_model="f1",             # 최고 모델을 판단할 기준 메트릭 (f1 점수)
    greater_is_better=True,                 # 점수가 높을수록 좋은 방향 (True → f1 높을수록 좋음)

    # --------------------------------------------------------
    # (6) 하드웨어 설정
    # --------------------------------------------------------
    fp16=torch.cuda.is_available(),         # GPU 사용 시 16-bit(FP16) 혼합정밀도 학습 활성화
    use_mps_device=(torch.backends.mps.is_available() 
                    if not torch.cuda.is_available() else False),  
                                            # Mac M1/M2 등 Apple Silicon(MPS) 가속기 사용 여부

    # --------------------------------------------------------
    # (7) 로깅 / WandB / TensorBoard 보고용 설정
    # --------------------------------------------------------
    report_to=[]                            # 외부 로깅 도구(W&B, TensorBoard 등) 비활성화
)

In [30]:
# ------------------------------------------------------------
# Trainer : 모델 학습을 자동으로 관리해주는 Hugging Face 고수준 API
# ------------------------------------------------------------
trainer = Trainer(
    model=model,                   # 학습시킬 모델 (여기서는 BertClsHead)
    args=args,                     # 위에서 정의한 TrainingArguments 설정
    train_dataset=train_tok,       # 학습용 데이터셋 (Dataset 형태)
    eval_dataset=test_tok,         # 평가용 데이터셋
    tokenizer=tokenizer,           # 토크나이저 (로깅/평가 시 필요)
    compute_metrics=metrics,       # 평가 시 사용할 메트릭 함수 (accuracy, f1 등)
)


C:\Users\ekfla\AppData\Local\Temp\ipykernel_25884\1455795040.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [32]:
# ============================================================
# 7. 평가 및 예측 테스트
# ============================================================

# ------------------------------------------------------------
# (1) 평가 (validation/test 데이터로 모델 성능 확인)
# ------------------------------------------------------------
# trainer.evaluate()는 test_tok(평가용 데이터셋)을 이용해
# model.forward()를 실행하고 metrics 함수(accuracy, f1)를 계산함
eval_res = trainer.evaluate()
print("평가 결과:", eval_res)
# 출력 예시: {'eval_loss': 0.23, 'eval_accuracy': 0.91, 'eval_f1': 0.90, ...}

c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


평가 결과: {'eval_loss': 0.7114917039871216, 'eval_model_preparation_time': 0.0014, 'eval_accuracy': 0.525, 'eval_f1': 0.6619217081850534, 'eval_runtime': 4.1251, 'eval_samples_per_second': 48.484, 'eval_steps_per_second': 3.151}


In [33]:
# ------------------------------------------------------------
# (2) 새 문장에 대한 예측 테스트
# ------------------------------------------------------------
# 감정분석용 샘플 문장 2개
samples = ["정말 감동적인 영화였습니다.", "지루하고 시간 낭비였어요."]

# ------------------------------------------------------------
# (3) 입력 문장 토크나이징 (KoBERT 입력 형식으로 변환)
# ------------------------------------------------------------
enc = tokenizer(
    samples,                        # 입력 문장 리스트
    return_tensors="pt",             # PyTorch 텐서 형태로 반환
    padding=True,                    # 배치 내 문장 길이 맞추기 (패딩 자동 추가)
    truncation=True                  # 최대 길이 초과 시 잘라냄
).to(model.classifier.weight.device)  # 모델이 올라간 디바이스(GPU/MPS/CPU)로 이동


In [34]:
# ------------------------------------------------------------
# (4) 모델 추론 (예측)
# ------------------------------------------------------------
# torch.no_grad() : 추론(inference) 시에는 gradient 계산 비활성화 → 메모리 절약, 속도 향상
with torch.no_grad():
    out = model(**enc)               # 모델에 입력 전달 (forward 수행)
    probs = torch.softmax(           # 출력 logits을 확률로 변환 (0~1 사이)
        out["logits"], dim=-1
    ).cpu().numpy()                  # GPU → CPU로 이동 후 NumPy 배열로 변환


In [35]:
# ------------------------------------------------------------
# (5) 예측 결과 출력
# ------------------------------------------------------------
# probs: 각 문장의 [부정 확률, 긍정 확률] 형태의 배열
for s, p in zip(samples, probs):
    print(f"[{s}] → 부정={p[0]:.3f}, 긍정={p[1]:.3f}, 예측={p.argmax()}")
    # p.argmax() : 확률이 더 큰 쪽(0=부정, 1=긍정)을 최종 예측으로 선택


[정말 감동적인 영화였습니다.] → 부정=0.428, 긍정=0.572, 예측=1
[지루하고 시간 낭비였어요.] → 부정=0.611, 긍정=0.389, 예측=0
